# Telco Müşteri Ayrılma (Churn) Tahmini

**Türkiye Yapay Zeka Akademisi — Makine Öğrenmesi Final Ödevi**

**Ad Soyad:** Yunus Büyükkafes

---

**Amaç:** Telekomünikasyon müşterilerinin hizmeti bırakıp bırakmayacağını (Churn) sınıflandırma modelleriyle tahmin etmek.

**Kütüphaneler:** pandas, numpy, scikit-learn, matplotlib, seaborn

**Veri seti:** IBM Telco Customer Churn (~500 satır, 21 sütun)

## 1. Kütüphanelerin Yüklenmesi

In [ ]:
!pip install -q pandas numpy scikit-learn matplotlib seaborn

import csv
import re
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif, VarianceThreshold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    f1_score, precision_score, recall_score,
)
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore")
%matplotlib inline
sns.set_theme(style="whitegrid", context="notebook")
RANDOM_STATE = 42

## 2–3. Veri Seti Yükleme ve Problem Tanımı

**Problem:** Telekomünikasyon müşterilerinin churn (ayrılma) durumunu tahmin etmek.

**Problem türü:** Sınıflandırma (ikili — Yes/No)

**Hedef değişken:** `Churn`

Aşağıdaki hücrede CSV dosyasını yükleyin: `WA_Fn-UseC_-Telco-Customer-Churn.csv`

In [ ]:
from google.colab import files

print("CSV dosyasını seçin...")
uploaded = files.upload()
DATA_PATH = Path(list(uploaded.keys())[0])
print(f"Yüklenen dosya: {DATA_PATH.name}")

def _fix_european_number(token: str) -> str:
    token = token.strip().strip('"')
    if re.fullmatch(r"\d+,\d+", token):
        digits = token.replace(",", "")
        return f"{digits[:-2]}.{digits[-2:]}" if len(digits) > 2 else f"0.{digits.zfill(2)}"
    return token

def load_telco_csv(path: Path) -> pd.DataFrame:
    with open(path, encoding="utf-8") as f:
        header = next(csv.reader([f.readline().strip()]))
        records = []
        for raw in f:
            line = raw.strip()
            if not line:
                continue
            if line.startswith('"') and line.endswith('"'):
                inner = line[1:-1]
                inner = re.sub(
                    r'""(\d+,\d+)""',
                    lambda m: _fix_european_number(m.group(1)),
                    inner,
                )
                inner = inner.replace('""', '"')
                parts = next(csv.reader([inner]))
            else:
                parts = next(csv.reader([line]))
            if len(parts) != len(header):
                continue
            parts[18] = _fix_european_number(parts[18])
            parts[19] = _fix_european_number(parts[19])
            records.append(parts)
    return pd.DataFrame(records, columns=header)

df = load_telco_csv(DATA_PATH)
print(f"Veri başarıyla okundu: {len(df)} satır, {len(df.columns)} sütun")

## 4. Temel Veri İnceleme (EDA)

In [ ]:
print("İlk 5 satır:")
display(df.head())
print(f"\nSatır-sütun: {df.shape[0]} satır, {df.shape[1]} sütun")
print("\nVeri tipleri:")
display(df.dtypes.to_frame("dtype"))
print("\nTemel istatistikler:")
display(df.describe(include="all").T)

df["SeniorCitizen"] = pd.to_numeric(df["SeniorCitizen"], errors="coerce")
df["tenure"] = pd.to_numeric(df["tenure"], errors="coerce")
df["MonthlyCharges"] = pd.to_numeric(df["MonthlyCharges"], errors="coerce")
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

print("\nSayısal sütun istatistikleri:")
display(df[["SeniorCitizen", "tenure", "MonthlyCharges", "TotalCharges"]].describe())
print("\nHedef dağılımı (Churn):")
display(df["Churn"].value_counts().to_frame("adet"))
display(df["Churn"].value_counts(normalize=True).round(3).to_frame("oran"))

## 5. Eksik Değer Kontrolü ve Temizleme

In [ ]:
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() else "Eksik değer yok (NaN).")
print(f"Toplam eksik hücre: {int(df.isnull().sum().sum())}")

if df["TotalCharges"].isnull().any():
    median_tc = df["TotalCharges"].median()
    n_miss = int(df["TotalCharges"].isnull().sum())
    df["TotalCharges"] = df["TotalCharges"].fillna(median_tc)
    print(f"TotalCharges: {n_miss} eksik değer medyan ({median_tc:.2f}) ile dolduruldu.")

df = df.drop(columns=["customerID"])
before = len(df)
df = df.dropna().reset_index(drop=True)
print(f"Satır sayısı: {before} -> {len(df)}")

## 7. Aykırı Değer İnceleme (IQR + Winsorize)

In [ ]:
numeric_cols = ["tenure", "MonthlyCharges", "TotalCharges"]

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, col in zip(axes, numeric_cols):
    sns.boxplot(y=df[col], ax=ax, color="#4C78A8")
    ax.set_title(col)
plt.tight_layout()
plt.show()

for col in numeric_cols:
    q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_out = int(((df[col] < low) | (df[col] > high)).sum())
    print(f"{col}: IQR dışı {n_out} gözlem | sınırlar=[{low:.2f}, {high:.2f}]")
    df[col] = df[col].clip(lower=low, upper=high)
print("Aykırı değerler IQR sınırlarına kırpıldı.")

## 9. Öznitelik Mühendisliği (Feature Engineering)

In [ ]:
df["AvgChargesPerMonth"] = np.where(
    df["tenure"] > 0, df["TotalCharges"] / df["tenure"], df["MonthlyCharges"]
)
service_cols = [
    "OnlineSecurity", "OnlineBackup", "DeviceProtection",
    "TechSupport", "StreamingTV", "StreamingMovies",
]
df["NumExtraServices"] = df[service_cols].apply(
    lambda row: sum(v == "Yes" for v in row), axis=1
)
df["TenureGroup"] = pd.cut(
    df["tenure"], bins=[-0.1, 12, 36, 72], labels=["0-12", "13-36", "37-72"]
).astype(str)
df["ChargeLevel"] = pd.qcut(
    df["MonthlyCharges"], q=3, labels=["Low", "Mid", "High"]
).astype(str)

print("Üretilen öznitelikler:")
print(" - AvgChargesPerMonth, NumExtraServices, TenureGroup, ChargeLevel")
display(df[["AvgChargesPerMonth", "NumExtraServices", "TenureGroup", "ChargeLevel"]].head())

## 6. Kategorik Değişken Encoding

In [ ]:
target = "Churn"
y = (df[target] == "Yes").astype(int)
X = df.drop(columns=[target])

binary_map = {"Yes": 1, "No": 0, "Female": 0, "Male": 1}
for col in ["gender", "Partner", "Dependents", "PhoneService", "PaperlessBilling"]:
    X[col] = X[col].map(binary_map)

multi_cat = [
    "MultipleLines", "InternetService", "OnlineSecurity", "OnlineBackup",
    "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies",
    "Contract", "PaymentMethod", "TenureGroup", "ChargeLevel",
]
X = pd.get_dummies(X, columns=multi_cat, drop_first=True)
X = X.apply(pd.to_numeric, errors="coerce").fillna(0)
print(f"Encoding sonrası öznitelik sayısı: {X.shape[1]}")

## 10. Öznitelik Seçimi

In [ ]:
var_selector = VarianceThreshold(threshold=0.01)
X_var = pd.DataFrame(
    var_selector.fit_transform(X),
    columns=X.columns[var_selector.get_support()],
    index=X.index,
)
print(f"VarianceThreshold: {X.shape[1]} -> {X_var.shape[1]}")

k = min(20, X_var.shape[1])
kbest = SelectKBest(score_func=f_classif, k=k)
X_selected = pd.DataFrame(
    kbest.fit_transform(X_var, y),
    columns=X_var.columns[kbest.get_support()],
    index=X_var.index,
)
scores = pd.Series(kbest.scores_, index=X_var.columns).sort_values(ascending=False)
print(f"SelectKBest (k={k}) top öznitelikler:")
display(scores.head(k).round(2).to_frame("F-skoru"))

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(X_selected.corr(), cmap="coolwarm", center=0, ax=ax, square=True)
ax.set_title("Seçilen Öznitelikler Korelasyon Matrisi")
plt.tight_layout()
plt.show()
feature_names = list(X_selected.columns)

## 11. Train / Validation / Test Ayrımı

In [ ]:
X_temp, X_test, y_temp, y_test = train_test_split(
    X_selected, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=RANDOM_STATE, stratify=y_temp
)
print(f"Train: {X_train.shape[0]} | Val: {X_val.shape[0]} | Test: {X_test.shape[0]}")
print(f"Churn oranı: {y_train.mean():.3f} / {y_val.mean():.3f} / {y_test.mean():.3f}")

## 8. Ölçekleme (StandardScaler)

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)
print("StandardScaler ile ölçekleme tamamlandı.")

## 12–13. Model Eğitimi ve Validation Karşılaştırması

In [ ]:
models = {
    "Logistic Regression": (
        LogisticRegression(max_iter=1000, random_state=RANDOM_STATE), True
    ),
    "KNN": (KNeighborsClassifier(n_neighbors=5), True),
    "Decision Tree": (
        DecisionTreeClassifier(random_state=RANDOM_STATE, max_depth=5), False
    ),
    "Random Forest": (
        RandomForestClassifier(
            n_estimators=200, random_state=RANDOM_STATE, max_depth=8
        ),
        False,
    ),
}

val_results = []
for name, (model, use_scaled) in models.items():
    Xtr = X_train_scaled if use_scaled else X_train
    Xva = X_val_scaled if use_scaled else X_val
    model.fit(Xtr, y_train)
    pred = model.predict(Xva)
    val_results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_val, pred),
        "Precision": precision_score(y_val, pred, zero_division=0),
        "Recall": recall_score(y_val, pred, zero_division=0),
        "F1": f1_score(y_val, pred, zero_division=0),
    })

results_df = pd.DataFrame(val_results).sort_values("F1", ascending=False)
display(results_df)
best_name = results_df.iloc[0]["Model"]
print(f"En iyi model (validation F1): {best_name}")

fig, ax = plt.subplots(figsize=(8, 4))
results_df.set_index("Model")[
    ["Accuracy", "Precision", "Recall", "F1"]
].plot(kind="bar", ax=ax, rot=15)
ax.set_ylim(0, 1)
ax.set_title("Validation Metrik Karşılaştırması")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

## 14. Hiperparametre Ayarlama (GridSearchCV)

In [ ]:
if best_name == "Random Forest":
    base = RandomForestClassifier(random_state=RANDOM_STATE)
    param_grid = {
        "n_estimators": [100, 200, 300],
        "max_depth": [4, 6, 8, None],
        "min_samples_split": [2, 5],
    }
    search_X, search_y, use_scaled_best = X_temp, y_temp, False
elif best_name == "Logistic Regression":
    base = LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)
    param_grid = {
        "C": [0.01, 0.1, 1, 10],
        "penalty": ["l2"],
        "solver": ["lbfgs"],
    }
    search_X, search_y = scaler.fit_transform(X_temp), y_temp
    use_scaled_best = True
elif best_name == "KNN":
    base = KNeighborsClassifier()
    param_grid = {
        "n_neighbors": [3, 5, 7, 11],
        "weights": ["uniform", "distance"],
        "p": [1, 2],
    }
    search_X, search_y = scaler.fit_transform(X_temp), y_temp
    use_scaled_best = True
else:
    base = DecisionTreeClassifier(random_state=RANDOM_STATE)
    param_grid = {
        "max_depth": [3, 5, 7, 10, None],
        "min_samples_split": [2, 5, 10],
        "criterion": ["gini", "entropy"],
    }
    search_X, search_y, use_scaled_best = X_temp, y_temp, False

grid = GridSearchCV(base, param_grid, cv=5, scoring="f1", n_jobs=-1, refit=True)
grid.fit(search_X, search_y)
print(f"En iyi parametreler: {grid.best_params_}")
print(f"CV F1: {grid.best_score_:.4f}")
best_model = grid.best_estimator_

scaler_final = StandardScaler()
X_temp_scaled = scaler_final.fit_transform(X_temp)
X_test_scaled_final = scaler_final.transform(X_test)

if use_scaled_best:
    best_model.fit(X_temp_scaled, y_temp)
    y_pred = best_model.predict(X_test_scaled_final)
else:
    best_model.fit(X_temp, y_temp)
    y_pred = best_model.predict(X_test)

## 15. Test Seti Değerlendirmesi

In [ ]:
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
cm = confusion_matrix(y_test, y_pred)

print(f"Model: {best_name}")
print(f"Accuracy: {acc:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f} | F1: {f1:.4f}")
print("\nConfusion Matrix:")
print(cm)
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["No Churn", "Churn"]))

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=["No Churn", "Churn"],
    yticklabels=["No Churn", "Churn"], ax=ax,
)
ax.set_xlabel("Tahmin")
ax.set_ylabel("Gerçek")
ax.set_title(f"Confusion Matrix — {best_name}")
plt.tight_layout()
plt.show()

## 17. Bonus — Model Açıklanabilirliği

In [ ]:
if hasattr(best_model, "feature_importances_"):
    imp = pd.Series(best_model.feature_importances_, index=feature_names)
    imp = imp.sort_values(ascending=False)
    print("Feature Importance (top 10):")
    display(imp.head(10).round(4).to_frame("importance"))
    fig, ax = plt.subplots(figsize=(8, 5))
    imp.head(10).sort_values().plot(kind="barh", ax=ax, color="#4C78A8")
    ax.set_title("En Önemli 10 Öznitelik")
    plt.tight_layout()
    plt.show()
    top_features = list(imp.head(5).index)
elif hasattr(best_model, "coef_"):
    coef = pd.Series(best_model.coef_.ravel(), index=feature_names)
    coef_abs = coef.reindex(coef.abs().sort_values(ascending=False).index)
    print("Logistic Regression katsayıları (top 10):")
    display(coef_abs.head(10).round(4).to_frame("katsayı"))
    fig, ax = plt.subplots(figsize=(8, 5))
    coef_abs.head(10).sort_values().plot(kind="barh", ax=ax, color="#4C78A8")
    ax.set_title("En Etkili 10 Katsayı")
    plt.tight_layout()
    plt.show()
    top_features = list(coef_abs.head(5).index)
else:
    print("KNN için SelectKBest skorları:")
    display(scores.head(10).round(2).to_frame("F-skoru"))
    top_features = list(scores.head(5).index)

## 16. Sonuç Yorumu

In [ ]:
print(f"""
ÖZET
====
- En iyi model (validation F1): {best_name}
- Test Accuracy={acc:.3f}, Precision={prec:.3f}, Recall={rec:.3f}, F1={f1:.3f}
- Öne çıkan değişkenler: {", ".join(top_features)}

YORUM:
- Sözleşme tipi, müşteri süresi (tenure) ve internet hizmeti churn tahmininde önemli.
- Kısa süreli ve aylık sözleşmeli müşteriler daha yüksek ayrılma riski taşır.
- Ek hizmet sayısı arttıkça churn olasılığı genelde azalır.

SINIRLILIKLAR:
- Veri seti ~500 satır; genelleme gücü sınırlı.
- Sınıf dengesizliği (churn ~%25) precision/recall dengesini etkiler.
- Harici ekonomik/rekabet faktörleri modelde yok.
""")
print("Proje başarıyla tamamlandı.")